DATA LOADING NOTEBOOK 
---
This notebook is intended for use in Google Colab or other cloud computing platforms. <br/> <br/>
It is solving the main obstacle we have currently with the dataset : its size. By streaming through the dataset instead of downloading it all once, we can do already do multiple analyses. <br/> <br/>
This is the main dataloader which creates the key part of the project : the co-commenting graph of all channels available in the dataset 


In [ ]:
!pip -q install fsspec aiohttp dask[dataframe] pyarrow


In [ ]:
import json, urllib.request, pandas as pd

record_id = "4650046"
with urllib.request.urlopen(f"https://zenodo.org/api/records/{record_id}") as r:
    rec = json.load(r)

def file_url(record_id, key):
    return f"https://zenodo.org/records/{record_id}/files/{key}?download=1"

rows = []
for f in rec["files"]:
    rows.append({
        "name": f["key"],
        "size_bytes": f["size"],
        "size_gb": round(f["size"] / (1024**3), 2),
        "url": file_url(record_id, f["key"]),
        "checksum_md5": f.get("checksum", "").replace("md5:", "")
    })

files = pd.DataFrame(rows).sort_values("size_bytes", ascending=False)
files.head(10)

In [ ]:
import pandas as pd, fsspec

def url_of(name):
    return files.loc[files["name"]==name, "url"].item()



LOAD CHANNEL METADATA :
Metadata of channels in the dataset

In [ ]:
name = "df_channels_en.tsv.gz"  
url = url_of(name)

with fsspec.open(url, "rb") as fh:
    dfChannels = pd.read_csv(fh, sep="\t", compression="gzip",
                     nrows=1_000_000, usecols=None)
dfChannels.head()


LOAD VIDEO METADATA FEATHER : <br/> <br/>
The Youniverse dataset holds two files for Youtube video metadata, the main one which contains all information and a lighter version which discards titles and descriptions. For this first task we only need to know which video is published by which channel



In [ ]:
import os, hashlib, requests, shutil, sys

URL = "https://zenodo.org/records/4650046/files/yt_metadata_helper.feather?download=1"
LOCAL = "/content/yt_metadata_helper.feather"


if not os.path.exists(LOCAL):
    with requests.get(URL, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        done = 0
        with open(LOCAL, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024*1024):
                if chunk:
                    f.write(chunk); done += len(chunk)
                    if total: print(f"\r{done/total:.1%}", end=""); sys.stdout.flush()




100.0%

In [ ]:
cols = ["display_id", "channel_id"]
videoDf = pd.read_feather(LOCAL, columns=cols, dtype_backend="pyarrow")

In [ ]:
videoDf.head()

In [ ]:
v2c = dict(zip(videoDf.display_id.values, videoDf.channel_id.values))


(OPTIONAL) : We can create a smaller graph by taking only big enough channels. We chose to do this filtering only after loading all the data

In [ ]:
channelSubset = dict(zip(dfChannels.channel.values, dfChannels.subscribers_cc.values > 200_000))

In [ ]:
channelSubsetDf = dfChannels[dfChannels['channel'].map(channelSubset)]
channelSubsetDf.to_csv('nodes.csv')

EDGE CREATION 
---
This is the main logic : we will linearly go through comment data in the dataset. The comment format is author | video_id | likes | replies. <br/> <br/>
For each author, we will see which channels he has mostly commented on. Then we will take the TOP_K_PER_AUTHOR most commented on channels and increment the weight of the edges between every combination of these channels. The logic behind this is that a single author may comment on a lot of channels without necessarily being part of their communities, taking only the top channels avoids this. <br/> This also lowers the bias towards bigger channels which users are drawn to in the beginning of their Youtube journeys. <br/> <br/>
The code also keeps track for every channel the number of total commenters, this will be used to normalize edge weights afterwards <br/> <br/>



In [ ]:
import pandas as pd
import fsspec
import pyarrow.feather as feather
from itertools import combinations
from collections import defaultdict, Counter
from tqdm import tqdm
import pickle, os, json, csv

### REFACTOR FILE PATHS IF NOT USING IN COLAB 


COMMENTS_URL = "youtube_comments.tsv.gz"

EDGES_OUT = "/content/channel_edges.csv"
CHUNKSIZE = 1_000_000
MAX_ROWS  = 300_000_000

TOP_K_PER_AUTHOR = 5     
MIN_CHANS_FOR_PAIRS = 2

AUTHOR_FLUSH_THRESHOLD = 500_000
CHECKPOINT_PATH = "/content/gdrive/My Drive/edges_checkpoint_2.pkl"
STATE_PATH      = "/content/gdrive/My Drive/state_2.json"
DICT_PATH       = "/content/gdrive/My Drive/dict_2.csv"
CHECKPOINT_EVERY = 5

if os.path.exists(CHECKPOINT_PATH):
    print("Resuming from checkpoint...")
    with open(CHECKPOINT_PATH, "rb") as f:
        edges_counter = pickle.load(f)
else:
    edges_counter = Counter()

if os.path.exists(DICT_PATH):
    channelToCommNumbers = defaultdict(int)
    with open(DICT_PATH, newline="") as f:
        r = csv.DictReader(f)
        for row in r:
            try:
                channelToCommNumbers[row["channel_id"]] = int(row["count"])
            except Exception:
                continue
else:
    channelToCommNumbers = defaultdict(int)

if os.path.exists(STATE_PATH):
    with open(STATE_PATH) as f:
        rows_seen = json.load(f)["rows_seen"]
else:
    rows_seen = 0

chunk_idx = 0
user_counts = defaultdict(Counter)
currentAuthor = None

def save_state_and_dict(rows_seen, channelToCommNumbers, dict_path=DICT_PATH, state_path=STATE_PATH):
    with open(dict_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["channel_id", "count"])
        w.writeheader()
        for ch, cnt in channelToCommNumbers.items():
            w.writerow({"channel_id": ch, "count": int(cnt)})
    with open(state_path, "w") as f:
        json.dump({"rows_seen": int(rows_seen)}, f)


def flush_authors(authors_dict):
    """
    For each author: keep only top-K channels (by count), then add +1 to each unordered pair.
    Also increment channelToCommNumbers for those top-K channels (unique author count).
    """
    global edges_counter, channelToCommNumbers
    for author, cnt in authors_dict.items():
        if not cnt:
            continue
        topk = [ch for ch, _ in cnt.most_common(TOP_K_PER_AUTHOR)]
        if len(topk) >= MIN_CHANS_FOR_PAIRS:
            for a, b in combinations(sorted(topk), 2):
                edges_counter[(a, b)] += 1
        for ch in topk:
            channelToCommNumbers[ch] += 1
    authors_dict.clear()

with fsspec.open(url_of(COMMENTS_URL), "rb") as fh:
    reader = pd.read_csv(
        fh,
        sep="\t",
        compression="gzip",
        usecols=["author", "video_id"],
        dtype=str,
        chunksize=CHUNKSIZE
    )

    for chunk in tqdm(reader, desc="Streaming comments in chunks"):
        chunk_idx += 1

        if MAX_ROWS and rows_seen >= MAX_ROWS:
            break

        if MAX_ROWS:
            remaining = MAX_ROWS - rows_seen
            if len(chunk) > remaining:
                chunk = chunk.iloc[:remaining]

        chunk["channel_id"] = chunk["video_id"].map(v2c)
        chunk = chunk.dropna(subset=["author", "channel_id"])

        for author, series in chunk.groupby("author")["channel_id"]:
            vc = series.value_counts()
            user_counts[author].update(vc.to_dict())
            currentAuthor = author

        rows_seen += len(chunk)

        if len(user_counts) > AUTHOR_FLUSH_THRESHOLD:
            flush_authors(user_counts)

        if chunk_idx % CHECKPOINT_EVERY == 0:
          with open(CHECKPOINT_PATH, "wb") as f:
              pickle.dump(edges_counter, f)

          temp_df = pd.DataFrame(((a, b, w) for (a, b), w in edges_counter.items()),
                                columns=["src", "dst", "weight"])
          temp_df.to_csv('edges_checkpoint.csv', index=False)

          save_state_and_dict(rows_seen, channelToCommNumbers)
          print(f"current Author = {currentAuthor}")
          print(f"[checkpoint] rows_seen={rows_seen:,}, authors_tracked={len(user_counts):,}, unique_edges={len(edges_counter):,}")

flush_authors(user_counts)

save_state_and_dict(rows_seen, channelToCommNumbers)


print(f"Processed ~{rows_seen:,} comment rows; unique edges: {len(edges_counter):,}")

edges_df = pd.DataFrame(((a, b, w) for (a, b), w in edges_counter.items()),
                        columns=["src", "dst", "weight"])
edges_df.to_csv(EDGES_OUT, index=False)
print(f"Wrote edges → {EDGES_OUT}")
print(f"Last author seen = {currentAuthor}")


OTHER APPROACH : TEMPORAL GRAPH 
---

We will define a start and end date for our graph, then we will filter videos to take only those which fall into our range. This will allow us to take into account only comments for this specific range and create a new channel graph.

In [ ]:
START_DATE = "2005-01-01"
END_DATE = "2010-12-31"
videoDf_filtered = videoDf[
        (videoDf["upload_date"] >= pd.to_datetime(START_DATE)) &
        (videoDf["upload_date"] <= pd.to_datetime(END_DATE))
    ]
v2c = dict(zip(videoDf_filtered.display_id.values, videoDf_filtered.channel_id.values))

In [ ]:
### SAME CELL

COMMENTS_URL = "youtube_comments.tsv.gz"

EDGES_OUT = "/content/channel_edges_range.csv"
CHUNKSIZE = 1_000_000
MAX_ROWS  = 1_000_000

TOP_K_PER_AUTHOR = 5     
MIN_CHANS_FOR_PAIRS = 2

# memory/flush & checkpoint
AUTHOR_FLUSH_THRESHOLD = 10_000
CHECKPOINT_PATH = "/content/gdrive/My Drive/edges_checkpoint_range.pkl"
STATE_PATH      = "/content/gdrive/My Drive/state_range.json"
DICT_PATH       = "/content/gdrive/My Drive/dict_range.csv"
CHECKPOINT_EVERY = 5

# resume edge counter
if os.path.exists(CHECKPOINT_PATH):
    print("Resuming from checkpoint...")
    with open(CHECKPOINT_PATH, "rb") as f:
        edges_counter = pickle.load(f)
else:
    edges_counter = Counter()

if os.path.exists(DICT_PATH):
    channelToCommNumbers = defaultdict(int)
    with open(DICT_PATH, newline="") as f:
        r = csv.DictReader(f)
        for row in r:
            try:
                channelToCommNumbers[row["channel_id"]] = int(row["count"])
            except Exception:
                continue
else:
    channelToCommNumbers = defaultdict(int)

if os.path.exists(STATE_PATH):
    with open(STATE_PATH) as f:
        rows_seen = json.load(f)["rows_seen"]
else:
    rows_seen = 0

chunk_idx = 0
user_counts = defaultdict(Counter)
currentAuthor = None

def save_state_and_dict(rows_seen, channelToCommNumbers, dict_path=DICT_PATH, state_path=STATE_PATH):
    # write the channel->unique_author_count dict
    with open(dict_path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["channel_id", "count"])
        w.writeheader()
        for ch, cnt in channelToCommNumbers.items():
            w.writerow({"channel_id": ch, "count": int(cnt)})
    # write the streaming state
    with open(state_path, "w") as f:
        json.dump({"rows_seen": int(rows_seen)}, f)


def flush_authors(authors_dict):
    """
    For each author: keep only top-K channels (by count), then add +1 to each unordered pair.
    Also increment channelToCommNumbers for those top-K channels (unique author count).
    """
    global edges_counter, channelToCommNumbers
    for author, cnt in authors_dict.items():
        if not cnt:
            continue
        # top-K channels this author commented on the most
        topk = [ch for ch, _ in cnt.most_common(TOP_K_PER_AUTHOR)]
        print(topk)
        if len(topk) >= MIN_CHANS_FOR_PAIRS:
            # +1 per unordered pair for this author
            for a, b in combinations(sorted(topk), 2):
                edges_counter[(a, b)] += 1
        for ch in topk:
            channelToCommNumbers[ch] += 1
    authors_dict.clear()

with fsspec.open(url_of(COMMENTS_URL), "rb") as fh:
    reader = pd.read_csv(
        fh,
        sep="\t",
        compression="gzip",
        usecols=["author", "video_id"],
        dtype=str,
        chunksize=CHUNKSIZE
    )

    for chunk in tqdm(reader, desc="Streaming comments in chunks"):
        chunk_idx += 1

        if MAX_ROWS and rows_seen >= MAX_ROWS:
            break

        if MAX_ROWS:
            remaining = MAX_ROWS - rows_seen
            if len(chunk) > remaining:
                chunk = chunk.iloc[:remaining]

        chunk["channel_id"] = chunk["video_id"].map(v2c)
        chunk = chunk.dropna(subset=["author", "channel_id"])

       
        for author, series in chunk.groupby("author")["channel_id"]:
            vc = series.value_counts()
            user_counts[author].update(vc.to_dict())
            
            currentAuthor = author

        rows_seen += len(chunk)

        if len(user_counts) > AUTHOR_FLUSH_THRESHOLD:
            flush_authors(user_counts)

        if chunk_idx % CHECKPOINT_EVERY == 0:
          with open(CHECKPOINT_PATH, "wb") as f:
              pickle.dump(edges_counter, f)

          temp_df = pd.DataFrame(((a, b, w) for (a, b), w in edges_counter.items()),
                                columns=["src", "dst", "weight"])
          temp_df.to_csv('edges_checkpoint.csv', index=False)

          save_state_and_dict(rows_seen, channelToCommNumbers)
          print(f"current Author = {currentAuthor}")
          print(f"[checkpoint] rows_seen={rows_seen:,}, authors_tracked={len(user_counts):,}, unique_edges={len(edges_counter):,}")

flush_authors(user_counts)

save_state_and_dict(rows_seen, channelToCommNumbers)


print(f"Processed ~{rows_seen:,} comment rows; unique edges: {len(edges_counter):,}")

edges_df = pd.DataFrame(((a, b, w) for (a, b), w in edges_counter.items()),
                        columns=["src", "dst", "weight"])
edges_df.to_csv(EDGES_OUT, index=False)
print(f"Wrote edges → {EDGES_OUT}")
print(f"Last author seen = {currentAuthor}")